# InsightWatch — Automated Anomaly Detection Pipeline

This notebook contains the finalized pipeline used to process the
retail dataset, detect unusual revenue movements, generate AI-assisted
business insights, store results in PostgreSQL, and trigger anomaly alerts.

In [9]:
## Setup

from google.colab import drive, userdata

drive.mount('/content/drive')

import os
import json
import pandas as pd
import numpy as np

# Install database driver
!pip install -q psycopg[binary] google-genai pydantic

import psycopg

# Project paths
base_path = "/content/drive/MyDrive/InsightWatch"
file_path = f"{base_path}/raw/online_retail_II.xlsx"

# Load Neon database credentials
db_url = userdata.get("NEON_DATABASE_URL")

# Basic checks
assert os.path.exists(file_path), (
    f"Dataset not found: {file_path}"
)

assert db_url is not None, (
    "NEON_DATABASE_URL not found in Colab Secrets"
)

print("✓ Google Drive connected")
print("✓ Dataset found")
print("✓ Neon database URL loaded")
print("InsightWatch ready.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive connected
✓ Dataset found
✓ Neon database URL loaded
InsightWatch ready.


In [10]:
## Data Preparation & KPI Generation

# --------------------------------------------
# 1. Load raw data
# --------------------------------------------

df_09_10 = pd.read_excel(
    file_path,
    sheet_name="Year 2009-2010"
)

df_10_11 = pd.read_excel(
    file_path,
    sheet_name="Year 2010-2011"
)

df = pd.concat(
    [df_09_10, df_10_11],
    ignore_index=True
)

print("Raw rows:", len(df))


# --------------------------------------------
# 2. Cleaning + transaction flags
# --------------------------------------------

df_clean = df.copy()

df_clean["IsCancellation"] = (
    df_clean["Invoice"]
    .astype(str)
    .str.startswith("C")
)

df_clean["IsReturn"] = (
    df_clean["Quantity"] < 0
)

df_clean["IsZeroPrice"] = (
    df_clean["Price"] == 0
)

df_clean["IsBadDebt"] = (
    df_clean["Description"]
    .fillna("")
    .str.lower()
    .eq("adjust bad debt")
)

df_clean["Revenue"] = (
    df_clean["Quantity"] *
    df_clean["Price"]
)


# --------------------------------------------
# 3. Remove exact duplicates
# --------------------------------------------

df_clean = df_clean.drop_duplicates()


# --------------------------------------------
# 4. Define genuine sales
# --------------------------------------------

df_clean["IsSale"] = (
    (df_clean["Quantity"] > 0) &
    (df_clean["Price"] > 0) &
    (~df_clean["IsBadDebt"])
)


# --------------------------------------------
# 5. Daily sales metrics
# --------------------------------------------

df_clean["Date"] = (
    df_clean["InvoiceDate"].dt.date
)

sales = df_clean[
    df_clean["IsSale"]
].copy()

daily_sales = (
    sales
    .groupby("Date")
    .agg(
        Revenue=("Revenue", "sum"),
        Orders=("Invoice", "nunique"),
        UnitsSold=("Quantity", "sum"),
        Customers=("Customer ID", "nunique")
    )
    .reset_index()
)

daily_sales["AOV"] = (
    daily_sales["Revenue"] /
    daily_sales["Orders"]
)


# --------------------------------------------
# 6. Daily return metrics
# --------------------------------------------

returns = df_clean[
    (df_clean["Quantity"] < 0) &
    (df_clean["Price"] > 0)
].copy()

daily_returns = (
    returns
    .groupby("Date")
    .agg(
        ReturnedUnits=(
            "Quantity",
            lambda x: abs(x.sum())
        ),
        ReturnValue=(
            "Revenue",
            lambda x: abs(x).sum()
        ),
        CancellationCount=(
            "Invoice",
            lambda x:
            x[
                x.astype(str)
                .str.startswith("C")
            ].nunique()
        )
    )
    .reset_index()
)


# --------------------------------------------
# 7. Combine daily metrics
# --------------------------------------------

daily_metrics = daily_sales.merge(
    daily_returns,
    on="Date",
    how="left"
)

daily_metrics[
    [
        "ReturnedUnits",
        "ReturnValue",
        "CancellationCount"
    ]
] = (
    daily_metrics[
        [
            "ReturnedUnits",
            "ReturnValue",
            "CancellationCount"
        ]
    ].fillna(0)
)


# --------------------------------------------
# 8. Weekday-aware revenue baseline
# --------------------------------------------

daily_metrics["Date"] = pd.to_datetime(
    daily_metrics["Date"]
)

daily_metrics["DayOfWeek"] = (
    daily_metrics["Date"].dt.day_name()
)

daily_metrics["RevenueBaseline"] = (
    daily_metrics
    .groupby("DayOfWeek")["Revenue"]
    .transform(
        lambda x:
        x.rolling(8).mean().shift(1)
    )
)

daily_metrics["RevenueDeviation"] = (
    (
        daily_metrics["Revenue"] -
        daily_metrics["RevenueBaseline"]
    )
    / daily_metrics["RevenueBaseline"]
) * 100

daily_metrics["RevenueStd"] = (
    daily_metrics
    .groupby("DayOfWeek")["Revenue"]
    .transform(
        lambda x:
        x.rolling(8).std().shift(1)
    )
)

daily_metrics["RevenueZScore"] = (
    (
        daily_metrics["Revenue"] -
        daily_metrics["RevenueBaseline"]
    )
    / daily_metrics["RevenueStd"]
)

daily_metrics["RevenueAnomaly"] = (
    daily_metrics["RevenueZScore"].abs() >= 3
)


# --------------------------------------------
# 9. Supporting metric baselines
# --------------------------------------------

daily_metrics["OrdersBaseline"] = (
    daily_metrics
    .groupby("DayOfWeek")["Orders"]
    .transform(
        lambda x:
        x.rolling(8).mean().shift(1)
    )
)

daily_metrics["AOVBaseline"] = (
    daily_metrics
    .groupby("DayOfWeek")["AOV"]
    .transform(
        lambda x:
        x.rolling(8).mean().shift(1)
    )
)

daily_metrics["UnitsBaseline"] = (
    daily_metrics
    .groupby("DayOfWeek")["UnitsSold"]
    .transform(
        lambda x:
        x.rolling(8).mean().shift(1)
    )
)

daily_metrics["OrdersDeviationPct"] = (
    (
        daily_metrics["Orders"] -
        daily_metrics["OrdersBaseline"]
    )
    / daily_metrics["OrdersBaseline"]
) * 100

daily_metrics["AOVDeviationPct"] = (
    (
        daily_metrics["AOV"] -
        daily_metrics["AOVBaseline"]
    )
    / daily_metrics["AOVBaseline"]
) * 100

daily_metrics["UnitsDeviationPct"] = (
    (
        daily_metrics["UnitsSold"] -
        daily_metrics["UnitsBaseline"]
    )
    / daily_metrics["UnitsBaseline"]
) * 100

daily_metrics["UnitsPerOrder"] = (
    daily_metrics["UnitsSold"] /
    daily_metrics["Orders"]
)


# --------------------------------------------
# 10. Severity + direction
# --------------------------------------------

def calculate_severity(zscore):
    z = abs(zscore)

    if z >= 6:
        return "Critical"
    elif z >= 4:
        return "High"
    elif z >= 3:
        return "Medium"
    else:
        return None


daily_metrics["Severity"] = (
    daily_metrics["RevenueZScore"]
    .abs()
    .apply(calculate_severity)
)

daily_metrics["AnomalyDirection"] = (
    daily_metrics["RevenueDeviation"]
    .apply(
        lambda x:
        "Positive" if x > 0
        else "Negative"
    )
)


# --------------------------------------------
# 11. Partial-day detection
# --------------------------------------------

last_date = (
    df_clean["InvoiceDate"]
    .max()
    .date()
)

daily_metrics["is_partial_day"] = (
    daily_metrics["Date"].dt.date ==
    last_date
)


# --------------------------------------------
# 12. Anomaly contexts
# --------------------------------------------

anomalies = daily_metrics[
    daily_metrics["RevenueAnomaly"]
].copy()


def create_anomaly_context(row):

    return {
        "date": str(row["Date"].date()),
        "day": row["DayOfWeek"],

        "revenue": float(
            round(row["Revenue"], 2)
        ),

        "baseline_revenue": float(
            round(row["RevenueBaseline"], 2)
        ),

        "revenue_deviation_pct": float(
            round(row["RevenueDeviation"], 2)
        ),

        "revenue_zscore": float(
            round(row["RevenueZScore"], 2)
        ),

        "severity": row["Severity"],
        "direction": row["AnomalyDirection"],

        "orders": int(row["Orders"]),

        "orders_deviation_pct": float(
            round(row["OrdersDeviationPct"], 2)
        ),

        "customers": int(row["Customers"]),

        "units_sold": int(row["UnitsSold"]),

        "units_deviation_pct": float(
            round(row["UnitsDeviationPct"], 2)
        ),

        "aov": float(
            round(row["AOV"], 2)
        ),

        "aov_deviation_pct": float(
            round(row["AOVDeviationPct"], 2)
        ),

        "units_per_order": float(
            round(row["UnitsPerOrder"], 2)
        ),

        "returned_units": int(
            row["ReturnedUnits"]
        ),

        "return_value": float(
            round(row["ReturnValue"], 2)
        ),

        "cancellations": int(
            row["CancellationCount"]
        ),

        "is_partial_day": bool(
            row["is_partial_day"]
        )
    }


anomaly_contexts = [
    create_anomaly_context(row)
    for _, row in anomalies.iterrows()
]

anomaly_context_df = pd.DataFrame(
    anomaly_contexts
)


print()
print("InsightWatch analysis rebuilt")
print("Raw rows:", len(df))
print("Daily records:", len(daily_metrics))
print("Revenue anomalies:", len(anomalies))
print("Dataset ends:", last_date)


Raw rows: 1067371

InsightWatch analysis rebuilt
Raw rows: 1067371
Daily records: 604
Revenue anomalies: 33
Dataset ends: 2011-12-09


In [11]:
## AI-Assisted Insights

# Detected anomalies are enriched with business context and passed to an LLM to generate concise, structured explanations.

# The model is instructed to use observed metrics as facts and avoid presenting unsupported explanations as causes.


from typing import List
from pydantic import BaseModel
from google import genai
from google.colab import userdata
import json

class AnomalyInsight(BaseModel):
    summary: str
    primary_driver: str
    supporting_metrics: List[str]
    return_signal: str
    investigations: List[str]


GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)


def generate_insight(context):

    prompt = f"""
You are the business intelligence layer of an automated anomaly detection system.

Analyze this detected revenue anomaly:

{json.dumps(context, indent=2)}

Important rules:
- All monetary values are in GBP (£).
- The anomaly was detected by a statistical Python model.
- Treat the provided metrics as observed facts.
- Do not invent causes.
- Do not claim that an order was corporate, wholesale, fraudulent,
  or caused by a system error unless the data proves it.
- Possible explanations must be clearly framed as things to investigate.
- Keep the analysis concise and useful to a business analyst.

Determine:
1. A concise summary of what happened.
2. The metric that appears to be the primary driver.
3. The most important supporting metrics.
4. What the return/cancellation numbers indicate.
5. One or two sensible investigations for an analyst.
"""

    interaction = client.interactions.create(
        model="gemini-3.5-flash-lite",
        input=prompt,
        response_format={
            "type": "text",
            "mime_type": "application/json",
            "schema": AnomalyInsight.model_json_schema()
        }
    )

    return AnomalyInsight.model_validate_json(
        interaction.output_text
    )


In [12]:
### Anomaly Detection



with psycopg.connect(db_url) as conn:
    with conn.cursor() as cur:

        cur.execute("SELECT version();")
        db_version = cur.fetchone()[0]

        cur.execute(
            "SELECT COUNT(*) FROM daily_metrics;"
        )
        daily_count = cur.fetchone()[0]

        cur.execute(
            "SELECT COUNT(*) FROM anomaly_insights;"
        )
        insight_count = cur.fetchone()[0]

print("✓ Neon connected")
print("PostgreSQL:", db_version.split(",")[0])
print("Daily metrics in DB:", daily_count)
print("AI insights in DB:", insight_count)




ai_query = """
SELECT
    anomaly_date,
    summary,
    primary_driver,
    supporting_metrics,
    return_signal,
    investigations
FROM anomaly_insights
ORDER BY anomaly_date;
"""

with psycopg.connect(db_url) as conn:
    with conn.cursor() as cur:

        cur.execute(ai_query)
        rows = cur.fetchall()

ai_results = []

for row in rows:

    ai_results.append({
        "date": str(row[0]),
        "summary": row[1],
        "primary_driver": row[2],
        "supporting_metrics": row[3],
        "return_signal": row[4],
        "investigations": row[5]
    })

ai_results_df = pd.DataFrame(ai_results)




fix_view_sql = """
CREATE OR REPLACE VIEW insightwatch_dashboard AS
SELECT
    d.metric_date,
    d.day_of_week,
    d.revenue,

    NULLIF(
        d.baseline_revenue,
        'NaN'::numeric
    ) AS baseline_revenue,

    NULLIF(
        d.revenue_deviation_pct,
        'NaN'::numeric
    ) AS revenue_deviation_pct,

    NULLIF(
        d.revenue_zscore,
        'NaN'::numeric
    ) AS revenue_zscore,

    d.orders,
    d.customers,
    d.units_sold,
    d.aov,
    d.returned_units,
    d.return_value,
    d.cancellation_count,

    d.revenue_anomaly,
    d.severity,
    d.anomaly_direction,
    d.is_partial_day,

    a.primary_driver,
    a.summary,
    a.return_signal,
    a.supporting_metrics,
    a.investigations

FROM daily_metrics d

LEFT JOIN anomaly_insights a
    ON d.metric_date = a.anomaly_date;
"""

with psycopg.connect(db_url) as conn:
    with conn.cursor() as cur:
        cur.execute(fix_view_sql)




with psycopg.connect(db_url) as conn:
    with conn.cursor() as cur:

        cur.execute(
            "SELECT COUNT(*) "
            "FROM insightwatch_dashboard;"
        )

        dashboard_count = cur.fetchone()[0]


print()

print("InsightWatch session restored")
print("Daily metrics:", len(daily_metrics))
print("Anomalies:", len(anomalies))
print("AI insights:", len(ai_results))
print("Dashboard rows:", dashboard_count)


✓ Neon connected
PostgreSQL: PostgreSQL 18.6 (4e955f5) on aarch64-unknown-linux-gnu
Daily metrics in DB: 604
AI insights in DB: 20

InsightWatch session restored
Daily metrics: 604
Anomalies: 33
AI insights: 20
Dashboard rows: 604


In [13]:
# Create alert history table

create_alert_table_sql = """
CREATE TABLE IF NOT EXISTS alert_history (
    id SERIAL PRIMARY KEY,
    anomaly_date DATE UNIQUE NOT NULL,
    severity VARCHAR(20),
    anomaly_direction VARCHAR(20),
    revenue NUMERIC,
    revenue_deviation_pct NUMERIC,
    revenue_zscore NUMERIC,
    primary_driver TEXT,
    alert_status VARCHAR(20) DEFAULT 'PENDING',
    sent_at TIMESTAMP
);
"""

with psycopg.connect(db_url) as conn:
    with conn.cursor() as cur:
        cur.execute(create_alert_table_sql)

print("✓ alert_history table ready")

✓ alert_history table ready


In [14]:
# Register detected anomalies


register_anomalies_sql = """
INSERT INTO alert_history (
    anomaly_date,
    severity,
    anomaly_direction,
    revenue,
    revenue_deviation_pct,
    revenue_zscore,
    primary_driver
)
SELECT
    d.metric_date,
    d.severity,
    d.anomaly_direction,
    d.revenue,
    d.revenue_deviation_pct,
    d.revenue_zscore,
    a.primary_driver
FROM daily_metrics d
LEFT JOIN anomaly_insights a
    ON d.metric_date = a.anomaly_date
WHERE d.revenue_anomaly = TRUE
ON CONFLICT (anomaly_date) DO NOTHING;
"""

with psycopg.connect(db_url) as conn:
    with conn.cursor() as cur:
        cur.execute(register_anomalies_sql)

print("✓ Anomalies registered in alert history")


# Check result
with psycopg.connect(db_url) as conn:
    with conn.cursor() as cur:
        cur.execute(
            "SELECT COUNT(*) FROM alert_history;"
        )
        count = cur.fetchone()[0]

print("Alert history records:", count)

✓ Anomalies registered in alert history
Alert history records: 33


In [15]:
## 6. Automated Email Alerts

from google.colab import userdata

SENDER_EMAIL = userdata.get("INSIGHTWATCH_EMAIL")
APP_PASSWORD = userdata.get("INSIGHTWATCH_APP_PASSWORD")

assert SENDER_EMAIL, "INSIGHTWATCH_EMAIL secret not found"
assert APP_PASSWORD, "INSIGHTWATCH_APP_PASSWORD secret not found"

# For now, send the test to yourself
ALERT_RECIPIENT = SENDER_EMAIL

print("✓ Email configuration loaded")
print("Sender:", SENDER_EMAIL)
print("Recipient:", ALERT_RECIPIENT)

✓ Email configuration loaded
Sender: adpbansal@gmail.com
Recipient: adpbansal@gmail.com


In [17]:
# Finalize alert history

with psycopg.connect(db_url) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            UPDATE alert_history
            SET alert_status = 'HISTORICAL'
            WHERE alert_status = 'PENDING';
        """)

with psycopg.connect(db_url) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT alert_status, COUNT(*)
            FROM alert_history
            GROUP BY alert_status
            ORDER BY alert_status;
        """)
        status_counts = cur.fetchall()

print("✓ Alert history finalized")
print()

for status, count in status_counts:
    print(f"{status}: {count}")

✓ Alert history finalized

HISTORICAL: 32
SENT: 1


In [18]:
# Automated anomaly alert pipeline


def send_anomaly_alerts():

    query = """
    SELECT
        d.metric_date,
        d.severity,
        d.anomaly_direction,
        d.revenue,
        d.revenue_deviation_pct,
        d.revenue_zscore,
        a.primary_driver,
        a.summary,
        a.return_signal
    FROM daily_metrics d
    LEFT JOIN anomaly_insights a
        ON d.metric_date = a.anomaly_date
    LEFT JOIN alert_history h
        ON d.metric_date = h.anomaly_date
    WHERE d.revenue_anomaly = TRUE
      AND h.anomaly_date IS NULL
    ORDER BY d.metric_date;
    """

    with psycopg.connect(db_url) as conn:
        with conn.cursor() as cur:
            cur.execute(query)
            new_alerts = cur.fetchall()

    if not new_alerts:
        print("✓ No new anomalies requiring alerts.")
        return

    print(f"New anomalies found: {len(new_alerts)}")

    for alert in new_alerts:

        (
            anomaly_date,
            severity,
            direction,
            revenue,
            deviation,
            zscore,
            primary_driver,
            summary,
            return_signal
        ) = alert

        # Register anomaly first
        with psycopg.connect(db_url) as conn:
            with conn.cursor() as cur:
                cur.execute(
                    """
                    INSERT INTO alert_history (
                        anomaly_date,
                        severity,
                        anomaly_direction,
                        revenue,
                        revenue_deviation_pct,
                        revenue_zscore,
                        primary_driver,
                        alert_status
                    )
                    VALUES (%s,%s,%s,%s,%s,%s,%s,'PENDING')
                    ON CONFLICT (anomaly_date) DO NOTHING;
                    """,
                    (
                        anomaly_date,
                        severity,
                        direction,
                        revenue,
                        deviation,
                        zscore,
                        primary_driver
                    )
                )

        # Build email
        msg = EmailMessage()

        msg["Subject"] = (
            f"InsightWatch Anomaly Alert — "
            f"{anomaly_date} ({severity})"
        )

        msg["From"] = SENDER_EMAIL
        msg["To"] = ALERT_RECIPIENT

        msg.set_content(f"""
INSIGHTWATCH — BUSINESS ANOMALY ALERT

Date: {anomaly_date}
Severity: {severity}
Direction: {direction}

Revenue: £{revenue:,.2f}
Deviation: {deviation:.2f}%
Z-Score: {zscore:.2f}

Primary Driver:
{primary_driver or "No AI insight available"}

AI Summary:
{summary or "No summary available"}

Return Signal:
{return_signal or "No return analysis available"}

This alert was generated by the InsightWatch anomaly detection pipeline.
""")

        try:

            with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
                server.login(SENDER_EMAIL, APP_PASSWORD)
                server.send_message(msg)

            with psycopg.connect(db_url) as conn:
                with conn.cursor() as cur:
                    cur.execute(
                        """
                        UPDATE alert_history
                        SET alert_status = 'SENT',
                            sent_at = NOW()
                        WHERE anomaly_date = %s
                        """,
                        (anomaly_date,)
                    )

            print(f"✓ Alert sent: {anomaly_date}")

        except Exception as e:

            print(f"✗ Failed: {anomaly_date}")
            print("Error:", e)


send_anomaly_alerts()

✓ No new anomalies requiring alerts.


In [19]:
#  Final project verification

with psycopg.connect(db_url) as conn:
    with conn.cursor() as cur:

        cur.execute("SELECT COUNT(*) FROM daily_metrics;")
        daily_count = cur.fetchone()[0]

        cur.execute("""
            SELECT COUNT(*)
            FROM daily_metrics
            WHERE revenue_anomaly = TRUE;
        """)
        anomaly_count = cur.fetchone()[0]

        cur.execute("SELECT COUNT(*) FROM anomaly_insights;")
        ai_count = cur.fetchone()[0]

        cur.execute("SELECT COUNT(*) FROM alert_history;")
        alert_count = cur.fetchone()[0]

        cur.execute("""
            SELECT COUNT(*)
            FROM alert_history
            WHERE alert_status = 'SENT';
        """)
        sent_count = cur.fetchone()[0]

print("INSIGHTWATCH — FINAL VERIFICATION")
print(f"Daily metrics       : {daily_count}")
print(f"Detected anomalies  : {anomaly_count}")
print(f"AI insights         : {ai_count}")
print(f"Alert history       : {alert_count}")
print(f"Emails sent         : {sent_count}")


INSIGHTWATCH — FINAL VERIFICATION
Daily metrics       : 604
Detected anomalies  : 33
AI insights         : 20
Alert history       : 33
Emails sent         : 1
